<a href="https://colab.research.google.com/github/KijoSal-dev/MLOPs-Streamlit-app-Integration/blob/main/Salome_Kungu_CS_DA03_26054_WK9_MLOPS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Import Required Libraries

In [ ]:
# 📦 Import libraries
import pandas as pd
import pickle
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import r2_score, mean_squared_error

# 1. Load the California Housing Dataset

In [ ]:
# 1. Load dataset
X, y = fetch_california_housing(return_X_y=True, as_frame=True)

# View dataset shape
print("Dataset Shape:", X.shape)

# View first rows
X.head()

Dataset Shape: (20640, 8)


,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25


# Exploring the dataset

In [ ]:
# Check column names
print("Feature Names:")
print(X.columns)

# Check dataset information
X.info()

# Summary statistics
X.describe()

Feature Names:
Index(['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup',
       'Latitude', 'Longitude'],
      dtype='object')
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20640 entries, 0 to 20639
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   MedInc      20640 non-null  float64
 1   HouseAge    20640 non-null  float64
 2   AveRooms    20640 non-null  float64
 3   AveBedrms   20640 non-null  float64
 4   Population  20640 non-null  float64
 5   AveOccup    20640 non-null  float64
 6   Latitude    20640 non-null  float64
 7   Longitude   20640 non-null  float64
dtypes: float64(8)
memory usage: 1.3 MB


,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude
count,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000
mean,3.870671,28.639486,5.429000,1.096675,1425.476744,3.070655,35.631861,-119.569704
std,1.899822,12.585558,2.474173,0.473911,1132.462122,10.386050,2.135952,2.003532
min,0.499900,1.000000,0.846154,0.333333,3.000000,0.692308,32.540000,-124.350000
25%,2.563400,18.000000,4.440716,1.006079,787.000000,2.429741,33.930000,-121.800000
50%,3.534800,29.000000,5.229129,1.048780,1166.000000,2.818116,34.260000,-118.490000
75%,4.743250,37.000000,6.052381,1.099526,1725.000000,3.282261,37.710000,-118.010000
max,15.000100,52.000000,141.909091,34.066667,35682.000000,1243.333333,41.950000,-114.310000


# 2. Train-Test Split (80/20)

In [ ]:
# 2. Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Training Set Shape:", X_train.shape)
print("Test Set Shape:", X_test.shape)

Training Set Shape: (16512, 8)
Test Set Shape: (4128, 8)


# 3. Preprocessing Pipeline (Imputation + Scaling)

In [ ]:
# 3. Preprocessing: Imputation + Scaling for numerical features

numeric_features = X.columns  # all features are numeric

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])

#4. Combine Preprocessing with ColumnTransformer

In [ ]:
# 4. Combine preprocessing using ColumnTransformer

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features)
    ]
)

#5. Build the Machine Learning Pipeline

In [ ]:
# 5. Build pipeline: preprocessing + KNN

pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('knn', KNeighborsRegressor())
])

pipeline

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer()),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  Index(['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup',
       'Latitude', 'Longitude'],
      dtype='object'))])),
                ('knn', KNeighborsRegressor())])

# 6. Define Hyperparameter Grid

In [ ]:
# 6. Define hyperparameter grid

param_grid = {
    'knn__n_neighbors': [3, 5, 7, 9],
    'knn__weights': ['uniform', 'distance'],
    'knn__p': [1, 2]
}

# 7. Apply GridSearchCV (5-Fold Cross Validation)

In [ ]:
# 7. Apply GridSearchCV with 5-fold cross-validation

grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=5,
    scoring='r2',
    verbose=1,
    n_jobs=-1
)

# Train the Model

In [ ]:
# 8. Fit the model

grid_search.fit(X_train, y_train)

Fitting 5 folds for each of 16 candidates, totalling 80 fits


GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('preprocessor',
                                        ColumnTransformer(transformers=[('num',
                                                                         Pipeline(steps=[('imputer',
                                                                                          SimpleImputer()),
                                                                                         ('scaler',
                                                                                          StandardScaler())]),
                                                                         Index(['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup',
       'Latitude', 'Longitude'],
      dtype='object'))])),
                                       ('knn', KNeighborsRegressor())]),
             n_jobs=-1,
             param_grid={'knn__n_neighbors': [3, 5, 7, 9], 'knn__p': [1, 2],
                         'knn__weights': ['uniform', 'distance']},
             scoring='r2', verbose=1)

# Evaluate the Model

In [ ]:
import numpy as np
# 9. Evaluate on test set

best_model = grid_search.best_estimator_

y_pred = best_model.predict(X_test)

r2 = r2_score(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)

# Print the Results

In [ ]:
# 10. Print results

print("Best Parameters:", grid_search.best_params_)
print("Best CV R² Score:", grid_search.best_score_)

print("\nTest Performance:")
print("Test R² Score:", r2)
print("Test MSE:", mse)
print("Test RMSE:", rmse)

Best Parameters: {'knn__n_neighbors': 9, 'knn__p': 1, 'knn__weights': 'distance'}
Best CV R² Score: 0.731266870986164

Test Performance:
Test R² Score: 0.72210916268423
Test MSE: 0.3641506481894662
Test RMSE: 0.6034489607162036


# 8. Save the Final Pipeline

In [ ]:
# 11. Save the pipeline

with open('california_knn_pipeline.pkl', 'wb') as f:
    pickle.dump(best_model, f)

print(" Final pipeline saved to 'california_knn_pipeline.pkl'")

 Final pipeline saved to 'california_knn_pipeline.pkl'


In [ ]:
# Load saved model

with open('california_knn_pipeline.pkl', 'rb') as f:
    loaded_model = pickle.load(f)

# Predict using loaded model
sample_prediction = loaded_model.predict(X_test.iloc[:5])

print("Sample Predictions:", sample_prediction)

Sample Predictions: [0.50997325 0.93723566 4.42174044 2.70423338 2.70450624]


# 9. (OPTIONAL)

# Install Required Libraries

In [ ]:
!pip install fastapi uvicorn pyngrok nest-asyncio

Import Libraries and Load Model

Define Input Schema

Create Prediction Endpoint

Start the API Server with a Public URL


In [ ]:
import pickle
import pandas as pd
from fastapi import FastAPI
from pydantic import BaseModel
from pyngrok import ngrok
import uvicorn
import nest_asyncio
import asyncio
from contextlib import asynccontextmanager

nest_asyncio.apply()

# Load trained model globally (outside FastAPI app)
with open("california_knn_pipeline.pkl", "rb") as f:
    model = pickle.load(f)

class HouseFeatures(BaseModel):
    MedInc: float
    HouseAge: float
    AveRooms: float
    AveBedrms: float
    Population: float
    AveOccup: float
    Latitude: float
    Longitude: float

@asynccontextmanager
async def lifespan(app: FastAPI):
    # Set up ngrok tunnel
    ngrok.set_auth_token("3AswnkXBzyGzVN3SRXUQTfSqWyq_3bJuB2fxmHArTa6r9oEGk")
    public_url = ngrok.connect(8000)
    print(f"Public URL: {public_url}")
    yield
    # Clean up
    ngrok.disconnect(True)

# Create FastAPI app with lifespan
app = FastAPI(lifespan=lifespan)

@app.get("/")
async def home():
    return {"message": "California Housing Prediction API"}

@app.post("/predict")
async def predict(data: HouseFeatures):
    input_data = pd.DataFrame([data.dict()])
    prediction = model.predict(input_data)
    return {
        "predicted_house_value": float(prediction[0])
    }

if __name__ == "__main__":
    # Use uvicorn.Server directly to manage the event loop
    config = uvicorn.Config(app, host="0.0.0.0", port=8000, loop="asyncio")
    server = uvicorn.Server(config)

    # Run the server in the existing event loop patched by nest_asyncio
    loop = asyncio.get_event_loop()
    loop.run_until_complete(server.serve())

INFO:     Started server process [256]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


Public URL: NgrokTunnel: "https://probankruptcy-unintelligibly-jacquetta.ngrok-free.dev" -> "http://localhost:8000"
INFO:     102.208.164.138:0 - "GET / HTTP/1.1" 200 OK
INFO:     102.208.164.153:0 - "GET /docs HTTP/1.1" 200 OK
INFO:     102.208.164.153:0 - "GET /openapi.json HTTP/1.1" 200 OK
INFO:     102.208.164.153:0 - "POST /predict HTTP/1.1" 200 OK


/tmp/ipykernel_256/3469385444.py:46: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  input_data = pd.DataFrame([data.dict()])


INFO:     102.208.164.153:0 - "GET / HTTP/1.1" 200 OK


# **Executive Summary**

California Housing Price Prediction API is a production-ready machine learning service that transforms a trained KNN model into a globally accessible web API. The system accepts 8 housing features (income, location, demographics) and returns accurate median house value predictions via a secure HTTPS endpoint.


**Target Variable: Median house value (in $100,000s)**

**Features (8):**

- MedInc: median income in block group
- HouseAge: median house age in block group
- AveRooms: average number of rooms per household
- AveBedrms: average number of bedrooms per household
- Population: block group population
- AveOccup: average number of household members
- Latitude: block group latitude
- Longitude: block group longitude


**Key Business Value:**

Instant Deployment: Local model → Public API in <5 minutes

Zero Infrastructure: No servers, cloud setup, or DevOps required

Global Access: Ngrok tunnel enables worldwide API calls

Production Features: Input validation, auto-documentation, error handling

Status: Live and verified at https://probankruptcy-unintelligibly-jacquetta.ngrok-free.dev/ with confirmed external access.

Use Cases: Real estate apps, investment analysis, market research, location scouting.

# **Technical Implementation Guide**

Code Architecture & Core Predict Endpoint
The system loads the trained model once when starting up, so every prediction is instant. Incoming data gets checked automatically to make sure it's the right format before reaching the model.


**The main prediction code:**


@app.post("/predict")

async def predict(data: HouseFeatures):

    input_data = pd.DataFrame([data.dict()])  # Turn JSON into table format
    prediction = model.predict(input_data)    # Get house price prediction
    return {"predicted_house_value": float(prediction[0])}


**How a Prediction Works (Step by Step)**

1. User sends house data (income, location, etc.) as JSON
2. System checks data is correct (numbers, right order)
3. Converts JSON → table format (what model expects)
4. Model makes prediction → "House worth $452,600"
5. Returns result as clean JSON response

**Sample Usage**

curl -X POST "https://probankruptcy-unintelligibly-jacquetta.ngrok-free.dev/predict" \

-H "Content-Type: application/json" \

-d '{"MedInc":8.3,"HouseAge":41,"AveRooms":6.98,"AveBedrms":1.08,"Population":1157,"AveOccup":2.27,"Latitude":37.88,"Longitude":-122.23}'

**Response: **

{
  "predicted_house_value": 4.355404712448459
}


**Limitations:**

- Ngrok free tier expires (2hrs)
- Single machine (no auto-scaling)
- No authentication
- No monitoring